In [1]:
# Let's implement a multi-agent system. 

In [2]:
import os
import requests
from dotenv import load_dotenv

In [3]:
from langchain_deepseek import ChatDeepSeek
from langchain_core import tools
from langchain.agents import create_agent
from langchain_tavily import TavilySearch, TavilyExtract
from tavily import TavilyClient
from langchain.tools import tool
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware


In [4]:
load_dotenv()

True

In [5]:
personal_data = {
    "ineterests": [
        "studying in rainy weather under balcony", 
        "drinking tea and sweats",
        "walking and listening to music",
        "seeing happy people on street and passing their times",
        "early mornings in a cool weather",
        "afternoon, golden hour, purple sky",
        "street cats",
        "driving on a good road (road trip)"
    ],
    "deseases": ["high-functional autism of adults", "aura-migrane caused by harsh light"],
    "habits & diets": ["5000 steps per day", "8 hours study per day", "eating ice-cream and sewats"],
    "age": 23,
}

In [6]:
todo_prompt = f"I'm heading to a trip to Mahmood-Abad, Mazandaran acommidating in a cozy villa in a rich town with private access to sea-coast and close to woods, give me a list of tasks to do during next 5 days, knowing that {personal_data}"

In [7]:
weather_prompt = f"find the best weather for each of the given tasks and tell me why you chose that tasks to be done in that day according to the weather forecast."

In [8]:
scheduler_prompt = f"knowing the {personal_data}, weather info and recommended tasks to be done, give me a timeline of the tasks on each day of travel in the form of a table"

In [9]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [10]:
tavily_tool = TavilySearch(
    max_results=5,
    topic="general",
    time_range="year"
)

In [11]:
extract_tool = TavilyExtract(
    extract_depth="advanced",  # "basic" or "advanced"
    format="markdown",          # "markdown" or "text"
    include_images=False,
)

In [12]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage
from langgraph.runtime import Runtime

In [13]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """Model-based guardrail: Use an LLM to evaluate response safety."""

    def __init__(self):
        super().__init__()
        # Use a fast/cheap model for the safety check
        self.safety_model = init_chat_model("deepseek-chat") 

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the final AI response
        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a model to evaluate safety
        safety_prompt = f"""Evaluate if this response is safe and appropriate.
        Respond with only 'SAFE' or 'UNSAFE'.

        Response: {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            # Block/redact the unsafe response
            last_message.content = "I cannot provide that response. It violated our safety guidelines."
        
        return None

In [14]:
@tool("weather_tool", return_direct=True)
def weather_tool(city:str) -> dict:
    """returns the weather forecast for a given city"""
    weather_api_key = os.getenv("WEATHER_API_KEY")
    url = "http://api.weatherapi.com/v1/current.json"
    params = {
        "key": weather_api_key,
        "q": city
    }
    r = requests.get(url, params=params)

    if r.status_code == 200: 
        return r.json()
    else:
        return "I was desprete to fetch the weather data!"

def scheduler(weather_info: dict, todo_list: dict) -> dict: 
    """Returns a dictionary of tasks to be done according to the timeline of travel and by considering the weather forecast"""

    

In [15]:
state = {
    "request": "Schedule the todo list of my travel!",
    "todo-list": None, 
    "weather-forecast": None,
    "scheduler": None
}

In [16]:
# "deepseek-chat" is the deepseek model we're looking for, and now we're initializing the LLM model. 

In [17]:
llm = ChatDeepSeek(model = "deepseek-chat", temperature=0.7)

In [18]:
todo_agent = create_agent(
    model=llm,
    tools=[tavily_tool, extract_tool],
    system_prompt=todo_prompt, 
)

In [19]:
weather_agent = create_agent(
    model=llm, 
    tools=[weather_tool],
    system_prompt=weather_prompt
)

In [20]:
scheduler_agent = create_agent(
    model=llm, 
    tools = [],
    middleware=[SafetyGuardrailMiddleware()],
    system_prompt = scheduler_prompt
)

In [21]:
from langgraph.config import get_stream_writer

@tool("plan_tasks", description="Brainstorm tasks for the trip based on user's interests. Use for initial planning.")
def call_todo_agent(request: str) -> str:
    writer = get_stream_writer()
    writer({"type": "status", "message": "🧠 Brainstorming trip tasks..."})
    result = todo_agent.invoke({"messages": [{"role": "user", "content": request}]})
    writer({"type": "status", "message": "✅ Tasks planned."})
    return result["messages"][-1].content

@tool("check_weather", description="Analyze weather forecast and match tasks to best days. Use after tasks are planned.")
def call_weather_agent(todo_list: str) -> str:
    writer = get_stream_writer()
    writer({"type": "status", "message": "🌤 Checking weather forecast..."})
    result = weather_agent.invoke({"messages": [{"role": "user", "content": todo_list}]})
    writer({"type": "status", "message": "✅ Forecast analyzed."})
    return result["messages"][-1].content

@tool("schedule_itinerary", description="Produce a day-by-day timeline table. Use when both tasks and weather are ready.")
def call_scheduler_agent(inputs: str) -> str:
    writer = get_stream_writer()
    writer({"type": "status", "message": "📅 Building day-by-day itinerary..."})
    result = scheduler_agent.invoke({"messages": [{"role": "user", "content": inputs}]})
    writer({"type": "status", "message": "✅ Itinerary ready."})
    return result["messages"][-1].content

In [22]:
# Define middleware once, reuse it
supervisor_middleware = [
    SafetyGuardrailMiddleware(),
    ModelRetryMiddleware(max_retries=3, backoff_factor=2.0),
    ModelCallLimitMiddleware(run_limit=10),
]

In [23]:
supervisor = create_agent(
    model=llm,
    tools=[call_todo_agent, call_weather_agent, call_scheduler_agent],
    system_prompt=(
        "You coordinate a trip-planning team. "
        "1. First call plan_tasks to get the task list. "
        "2. Then call check_weather with those tasks. "
        "3. Finally call schedule_itinerary with both outputs. "
        "Return only the final itinerary."
    ),
    middleware=supervisor_middleware,   # <-- use the list variable
)

In [24]:
result = supervisor.invoke({
    "messages": [{"role": "user", "content": todo_prompt}]
})
print(result["messages"][-1].content)

# 5-Day Mahmood-Abad Trip Itinerary

**Traveller:** 23 y/o · high-functioning autism · aura-migraine (harsh-light trigger) · 8h study/day · 5,000 steps/day

**Weather (2026-10-11):** 28.3°C, Sunny, **UV 8.2 (HIGH)**, cloud 5%, humidity 14%, wind 4.7 kph S, no rain forecast, visibility 10 km.
→ **Midday 11:00–15:30 = indoor/shaded only.** Outdoor windows: **06:00–08:00** and **16:30–19:00**.

---

## Day 1 — Arrival & Settle
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Early morning walk | Villa area | Shift to arrival time if later |
| 07:00–08:00 | Breakfast + tea | Villa | Unpack, orient territory |
| 08:00–12:00 | Study block 1 | Balcony (shaded) | Sunny — use umbrella/awning |
| 12:00–13:00 | Lunch | Villa | Indoor |
| 13:00–15:30 | Study block 2 | Indoor, shaded | **UV 8.2 peak — stay in** |
| 15:30–16:30 | Study block 3 | Indoor | |
| 16:30–19:00 | Coastal golden hour walk | Private beach access | Sunglasses + hat |
| 19:00–20:00 | Dinner | Villa / local |

In [25]:
from textwrap import fill
from IPython.display import display, Markdown, HTML

def show_stage(title: str, body: str, color: str = "#307fc1"):
    """Print a clean, boxed section header + body as rendered Markdown."""
    bar = "─" * 60
    header = f"<div style='color:{color};font-weight:600;font-size:1.05em'>{bar}<br>▶  {title}<br>{bar}</div>"
    display(HTML(header))
    display(Markdown(body if body else "_(empty)_"))
    print()  # spacing

In [27]:
show_stage(
    "FINAL ITINERARY",
    result["messages"][-1].content,
    color="#2e7d32",
)

# 5-Day Mahmood-Abad Trip Itinerary

**Traveller:** 23 y/o · high-functioning autism · aura-migraine (harsh-light trigger) · 8h study/day · 5,000 steps/day

**Weather (2026-10-11):** 28.3°C, Sunny, **UV 8.2 (HIGH)**, cloud 5%, humidity 14%, wind 4.7 kph S, no rain forecast, visibility 10 km.
→ **Midday 11:00–15:30 = indoor/shaded only.** Outdoor windows: **06:00–08:00** and **16:30–19:00**.

---

## Day 1 — Arrival & Settle
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Early morning walk | Villa area | Shift to arrival time if later |
| 07:00–08:00 | Breakfast + tea | Villa | Unpack, orient territory |
| 08:00–12:00 | Study block 1 | Balcony (shaded) | Sunny — use umbrella/awning |
| 12:00–13:00 | Lunch | Villa | Indoor |
| 13:00–15:30 | Study block 2 | Indoor, shaded | **UV 8.2 peak — stay in** |
| 15:30–16:30 | Study block 3 | Indoor | |
| 16:30–19:00 | Coastal golden hour walk | Private beach access | Sunglasses + hat |
| 19:00–20:00 | Dinner | Villa / local | |
| 20:00–21:30 | Tea, ice cream, music | Town / villa | Town ice-cream stop |

**Anchor:** Villa orientation + first beach walk.

---

## Day 2 — Sunrise Beach + Forest Park
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Sunrise beach walk | Private sea access | Cool, low UV — ideal |
| 07:00–08:00 | Breakfast + tea | Villa | |
| 08:00–12:00 | Study block 1 | Balcony | |
| 12:00–13:00 | Lunch | Villa | |
| 13:00–15:30 | Study block 2 | Indoor shaded | Avoid midday sun |
| 15:30–16:30 | Study block 3 | Indoor | |
| 16:30–19:00 | Hezar Pelleh Forest Park walk | Forest Park | Shaded canopy — migraine-safe |
| 19:00–20:00 | Dinner | Villa / local | |
| 20:00–21:30 | Tea, sweets, music | Villa | Watch for street cats 🐈 |

**Anchor:** Hezar Pelleh Forest Park shaded walk.

---

## Day 3 — Road Trip Day
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Early morning walk | Coastal road | Before departure |
| 07:00–08:00 | Breakfast + tea | Villa | Pack snacks/water |
| 08:00–12:00 | Study block 1 | Balcony (early) | Shortened if departing early |
| 12:00–13:00 | Lunch | En route | Fereydunkenar–Babolsar |
| 13:00–15:30 | Coastal drive | Fereydunkenar → Babolsar | **In-car shaded; sunglasses essential** |
| 15:30–16:30 | Study block 3 | Café / villa return | Light session |
| 16:30–19:00 | Promenade people-watching | Babolsar | Golden hour — favourite interest |
| 19:00–20:00 | Dinner | Babolsar | |
| 20:00–21:30 | Ice cream + music drive back | Coastal road | Good road + music = ideal |

**Anchor:** Fereydunkenar–Babolsar coastal road trip. *(Study reduced today → compensate on Day 4.)*

---

## Day 4 — Slow Rainy Study Day (flexible)
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Early morning walk | Beach / covered path | If rain: covered walk or short drive |
| 07:00–08:00 | Breakfast + tea | Villa | |
| 08:00–12:00 | Study block 1 (extended) | Balcony | **Rain ideal — bonus scenario, not guaranteed** |
| 12:00–13:00 | Lunch | Villa | |
| 13:00–15:30 | Study block 2 (extended) | Indoor shaded | Low-stimulation day |
| 15:30–16:30 | Study block 3 | Indoor | |
| 16:30–19:00 | Covered walk / scenic drive | Villa area / coast | If sunny: shaded route only |
| 19:00–20:00 | Dinner | Villa | |
| 20:00–21:30 | Tea, ice cream, music | Villa | |

**Anchor:** Full balcony study day (rain-dependent bonus). *(Extra study time to hit the 8h/day average.)*

---

## Day 5 — Final Sunrise & Wrap-Up
| Time | Task | Location | Notes |
|---|---|---|---|
| 06:00–07:00 | Final sunrise beach walk | Private sea access | |
| 07:00–08:00 | Breakfast + tea | Villa | Begin packing |
| 08:00–12:00 | Study block 1 | Balcony | |
| 12:00–13:00 | Lunch | Villa | |
| 13:00–15:30 | Study block 2 | Indoor shaded | |
| 15:30–16:30 | Study block 3 | Indoor | |
| 16:30–19:00 | Golden hour finale walk | Beach / forest edge | Last street-cat spotting 🐈 |
| 19:00–20:00 | Dinner | Villa / local | |
| 20:00–21:30 | Tea, sweets, music + wrap-up | Villa | |

**Anchor:** Golden hour finale walk.

---

## Daily Health & Habit Checklist
| Item | Target | How |
|---|---|---|
| Steps | 5,000/day | Morning walk (~2,500) + golden hour walk (~2,500) |
| Study | 8h/day | 08:00–12:00 + 13:00–15:30 + 15:30–16:30 (Day 3 shortened → Day 4 extended) |
| Migraine protection | Avoid 11:00–15:30 sun | Indoor/shaded blocks; sunglasses + hat outdoors |
| Autism-friendly rhythm | Fixed schedule | Same times daily; one anchor activity/day |
| Comfort interests | Daily | Tea, ice cream/sweets, music, street cats, golden hour |

**Weather contingency:** No rain in current forecast — treat Day 4 balcony-rain study as a bonus. If rain arrives, swap any outdoor block for a covered walk or scenic drive. If UV stays high, keep all 11:00–15:30 activity indoors regardless of the day plan.